# Highlight PII in a PDF using the ASE analyzer

This notebook extracts PDF text on the kernel host, sends text containers to your existing ASE analyzer over authenticated HTTPS, and writes highlight annotations locally. It does not run local Presidio engines or download spaCy models. The anonymizer service is not needed.

**Highlights are not redaction. The output retains the original PII and must be protected like the input.** Start with a synthetic PDF. Do not print extracted text, matched values, tokens, or raw HTTP error bodies. Clear outputs before sharing.

## Prerequisites

- Use the same Azure VM/kernel network path and managed identity as the working [ASE connection notebook](presidio_ase_notebook.ipynb). The VM must resolve the analyzer's private hostname and reach HTTPS port 443.
- Configure the analyzer API's registered Application ID URI and, for a user-assigned managed identity, its **client ID**. The App Service caller allowlists must accept the identity's client ID and principal/object ID in their respective fields. Keep the tenant restriction.
- Match App Service authentication to the API's token version. For the v1 Government tokens used by the working setup, the configured issuer URL is `https://login.microsoftonline.us/<tenant-id>/` (without `/v2.0`). Government v1 token claims normally contain `https://sts.windows.net/<tenant-id>/` as the issuer. Do not change cloud endpoints or disable authentication to work around an error.
- Keep TLS verification enabled. Configure `REQUESTS_CA_BUNDLE` on the kernel host if your organization uses a private CA.

Install dependencies **on the kernel host**, from a terminal, using your existing environment (Windows example):

```powershell
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel pdfminer.six pikepdf
```

Select that environment as the notebook kernel. Use Python 3.10 through 3.14 to stay within this repository's supported range. This notebook does not modify Azure resources or permissions.


In [ ]:
import math
import os
from pathlib import Path
from urllib.parse import urlsplit

import requests
from azure.core.exceptions import AzureError
from azure.identity import ManagedIdentityCredential
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTChar, LTTextContainer, LTTextLine
from pikepdf import Array, Dictionary, Name, Pdf

# Use the values from your working ASE connection notebook.
ANALYZER_BASE_URL = os.environ.get(
    "ANALYZER_BASE_URL",
    "https://presidio-analyzer.ase-jtl.appserviceenvironment.us",
).rstrip("/")
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "").strip()
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "").strip()
LANGUAGE = "en"
TIMEOUT = (10, 60)

# Paths are relative to the kernel's working directory; absolute paths also work.
INPUT_PDF = Path(os.environ.get(
    "INPUT_PDF", str(Path("..") / "python" / "sample_data" / "sample.pdf")
)).expanduser()
OUTPUT_PDF = Path(os.environ.get(
    "OUTPUT_PDF", str(INPUT_PDF.with_name(INPUT_PDF.stem + "_annotated.pdf"))
)).expanduser()


## Validate configuration and connect

Set `ANALYZER_API_RESOURCE` to the registered Application ID URI, not a guessed hostname or the Azure Resource Manager audience. Do not append `/.default`; the client does that. For system-assigned identity, leave `MANAGED_IDENTITY_CLIENT_ID` empty.

The default input path assumes the kernel's working directory is this notebook's `ASE` folder. Otherwise set `INPUT_PDF` to an absolute path. Set `OUTPUT_PDF` to a new file in an existing folder. To protect existing files, overwriting an input or output is refused, including on reruns.


In [ ]:
def validate_configuration() -> None:
    """Validate the service configuration and local PDF paths.

    :return: None.
    :raises ValueError: Invalid service settings or overlapping PDF paths.
    :raises FileNotFoundError: Missing input file or output folder.
    :raises FileExistsError: Output file already exists.
    """
    try:
        parsed = urlsplit(ANALYZER_BASE_URL)
        port = parsed.port
    except ValueError:
        raise ValueError("Invalid analyzer HTTPS base URL.") from None
    if (
        parsed.scheme != "https"
        or not parsed.hostname
        or parsed.username is not None
        or parsed.password is not None
        or parsed.path
        or parsed.query
        or parsed.fragment
        or port not in (None, 443)
    ):
        raise ValueError("Configure a root HTTPS base URL for the analyzer.")
    if not ANALYZER_API_RESOURCE or ANALYZER_API_RESOURCE.endswith("/.default"):
        raise ValueError("Set ANALYZER_API_RESOURCE to the registered URI without /.default.")
    if not INPUT_PDF.is_file():
        raise FileNotFoundError("INPUT_PDF must identify an existing PDF on the kernel host.")
    if INPUT_PDF.resolve() == OUTPUT_PDF.resolve():
        raise ValueError("INPUT_PDF and OUTPUT_PDF must be different files.")
    if OUTPUT_PDF.exists():
        raise FileExistsError("OUTPUT_PDF already exists; choose a new output file.")
    if not OUTPUT_PDF.parent.is_dir():
        raise FileNotFoundError("The OUTPUT_PDF parent folder must exist.")


validate_configuration()
credential = (
    ManagedIdentityCredential(client_id=MANAGED_IDENTITY_CLIENT_ID)
    if MANAGED_IDENTITY_CLIENT_ID
    else ManagedIdentityCredential()
)
session = requests.Session()
# Prevent .netrc credentials from replacing the bearer token.
session.auth = lambda request: request


def analyzer_request(path: str, payload: dict | None = None) -> requests.Response:
    """Call the analyzer without exposing tokens or response bodies.

    :param path: Analyzer endpoint path.
    :param payload: JSON body, or None for a GET request.
    :return: Successful HTTP response.
    :raises RuntimeError: Token acquisition, transport, or HTTP failure.
    """
    try:
        token = credential.get_token(ANALYZER_API_RESOURCE.rstrip("/") + "/.default")
    except (AzureError, ValueError):
        raise RuntimeError("Token acquisition failed; check the VM identity and API resource.") from None
    try:
        response = session.request(
            "GET" if payload is None else "POST",
            ANALYZER_BASE_URL + path,
            headers={"Authorization": "Bearer " + token.token},
            json=payload,
            timeout=TIMEOUT,
            allow_redirects=False,
            verify=True,
        )
    except requests.exceptions.SSLError:
        raise RuntimeError("TLS validation failed; check the trusted CA chain.") from None
    except requests.exceptions.Timeout:
        raise RuntimeError("Analyzer request timed out; check connectivity and service health.") from None
    except requests.exceptions.RequestException:
        raise RuntimeError("Analyzer request failed; check private DNS and HTTPS connectivity.") from None
    if response.status_code != 200:
        raise RuntimeError(
            f"Analyzer returned HTTP {response.status_code}; see troubleshooting below. "
            "Response body omitted to protect sensitive data."
        )
    return response


analyzer_request("/health")
print("Analyzer health endpoint returned HTTP 200.")


## Extract text, detect PII, and map offsets to PDF glyphs

Send exactly the extracted text without stripping or normalizing it. Analyzer offsets include spaces and newlines, even when those characters have no visible PDF glyph. Keep placeholders for those characters so highlights do not shift. Retain the page index and split multi-line detections into separate line highlights.

This example supports PDFs with extractable text. Scanned/image-only pages require OCR, which is not included. Extraction order, ligatures, unusual fonts, hidden content, or rotated text can affect detection and annotation accuracy. Inspect the result before relying on it; detection is not a guarantee that all PII was found.


In [ ]:
def validate_detections(data, text: str) -> list[dict]:
    """Validate REST detections before using offsets as PDF indices.

    :param data: Decoded analyzer response.
    :param text: Exact text sent to the analyzer.
    :return: Validated detection dictionaries.
    :raises RuntimeError: Unexpected response shape or invalid detection.
    """
    if not isinstance(data, list):
        raise RuntimeError("Analyzer response must be a list of detections.")
    for result in data:
        if not isinstance(result, dict):
            raise RuntimeError("Each analyzer detection must be an object.")
        start, end = result.get("start"), result.get("end")
        score, entity_type = result.get("score"), result.get("entity_type")
        if (
            type(start) is not int
            or type(end) is not int
            or not 0 <= start < end <= len(text)
            or type(score) not in (int, float)
            or not math.isfinite(score)
            or not 0 <= score <= 1
            or not isinstance(entity_type, str)
            or not entity_type.strip()
        ):
            raise RuntimeError("Analyzer returned invalid offsets, score, or entity type.")
    return data


def text_with_geometry(container: LTTextContainer) -> tuple[str, list]:
    """Map every text character to line geometry or a non-glyph placeholder.

    :param container: PDF text container.
    :return: Exact extracted text and one geometry entry per character.
    :raises RuntimeError: Unsupported layout or inconsistent text extraction.
    """
    parts, geometry = [], []
    for line_index, line in enumerate(container):
        if not isinstance(line, LTTextLine):
            raise RuntimeError("Unsupported PDF text layout; inspect extraction before annotating.")
        for item in line:
            item_text = item.get_text()
            parts.append(item_text)
            entry = (line_index, item.bbox) if isinstance(item, LTChar) else None
            geometry.extend([entry] * len(item_text))
    text = "".join(parts)
    if text != container.get_text():
        raise RuntimeError("PDF text and character geometry do not align; annotation stopped.")
    return text, geometry


def detection_boxes(result: dict, geometry: list) -> list[tuple]:
    """Combine detected glyph boxes separately for each PDF text line.

    :param result: Validated analyzer detection.
    :param geometry: Character-aligned geometry entries.
    :return: One bounding box per line with detected glyphs.
    :raises RuntimeError: A detection contains no visible glyphs.
    """
    lines = {}
    for entry in geometry[result["start"]:result["end"]]:
        if entry is None:
            continue
        line_index, box = entry
        if line_index in lines:
            previous = lines[line_index]
            box = (
                min(previous[0], box[0]), min(previous[1], box[1]),
                max(previous[2], box[2]), max(previous[3], box[3]),
            )
        lines[line_index] = box
    if not lines:
        raise RuntimeError("A detection has no visible PDF glyphs; inspect text extraction.")
    return list(lines.values())


In [ ]:
analyzed_bounding_boxes = []
detection_count = 0
text_container_count = 0
page_count = 0

for page_index, page_layout in enumerate(extract_pages(INPUT_PDF)):
    page_count += 1
    for container in page_layout:
        if not isinstance(container, LTTextContainer):
            continue
        text, geometry = text_with_geometry(container)
        if not text.strip():
            continue
        text_container_count += 1
        response = analyzer_request("/analyze", {"text": text, "language": LANGUAGE})
        try:
            data = response.json()
        except ValueError:
            raise RuntimeError("Analyzer returned invalid JSON; check the endpoint and image version.") from None
        results = validate_detections(data, text)
        detection_count += len(results)
        for result in results:
            for box in detection_boxes(result, geometry):
                analyzed_bounding_boxes.append({
                    "page_index": page_index,
                    "bounding_box": box,
                    "entity_type": result["entity_type"],
                })

if text_container_count == 0:
    raise RuntimeError("No extractable text found; scanned PDFs require a separate OCR workflow.")
print(f"Analyzed {page_count} pages and {text_container_count} text containers; {detection_count} detections.")


## Write highlight annotations

Append highlights to their original pages while preserving existing annotations. A zero-detection result produces a PDF with no new highlights; it does not certify that the document contains no PII. Input text, metadata, attachments, images, and hidden PDF content are not removed.


In [ ]:
validate_configuration()
with Pdf.open(INPUT_PDF) as pdf:
    if len(pdf.pages) != page_count:
        raise RuntimeError("PDF page counts disagree; annotation stopped.")
    for detected_box in analyzed_bounding_boxes:
        x0, y0, x1, y1 = detected_box["bounding_box"]
        highlight = Dictionary(
            Type=Name.Annot,
            Subtype=Name.Highlight,
            QuadPoints=[x0, y1, x1, y1, x0, y0, x1, y0],
            Rect=[x0, y0, x1, y1],
            C=[1, 0, 0],
            CA=0.5,
            T=detected_box["entity_type"],
        )
        page = pdf.pages[detected_box["page_index"]]
        if Name.Annots not in page:
            page.Annots = Array()
        page.Annots.append(pdf.make_indirect(highlight))
    pdf.save(OUTPUT_PDF)
print(f"Saved PDF with {len(analyzed_bounding_boxes)} new highlights. Original PII remains present.")


## Troubleshooting

| Failure | Check |
| --- | --- |
| Token acquisition | Kernel host identity endpoint; attached identity; user-assigned identity **client ID**; registered API resource. |
| HTTP 401/403 | API audience, token-version/issuer compatibility, caller application and identity allowlists, required app roles, and App Service network access restrictions. A 403 alone does not identify which layer rejected the request. |
| TLS or connection failure | Private DNS, VM-to-ASE routing, HTTPS port 443, proxy configuration, and trusted CA bundle. Do not disable TLS verification. |
| HTTP 400/413 | Supported language and the deployed analyzer's request/text-size limits. |
| File/path error | Files must exist on the kernel host; use absolute paths if needed. Choose a new output filename for each run. |
| No text or misplaced highlights | Scanned PDF, extraction order, unusual fonts, rotated text, or glyph alignment. Inspect results; use an OCR-aware workflow where needed. |

Close the HTTP session and credential after processing (rerun the client cell before making further requests).

For maintainers: from the repository root, run `python -m unittest discover -s docs/samples/ASE/tests -v` using an environment with the dependencies above. The tests use synthetic PDFs and mocked authentication/HTTP, not your live ASE.


In [ ]:
session.close()
credential.close()
